# Decision Tree Model for Air Quality Classification

This notebook trains a Decision Tree Classifier to predict air quality levels as High, Medium, or Low using the processed dataset generated from the preprocessing pipeline.

## 1. Load Processed Dataset

In [27]:
# Import required libraries
import pandas as pd
import json

# Load processed train, validation, and test datasets
# NOTE: "../" is used because this notebook is inside the "notebooks" folder
train = pd.read_parquet("../data/processed/train.parquet")
val = pd.read_parquet("../data/processed/val.parquet")
test = pd.read_parquet("../data/processed/test.parquet")

# Load metadata file
with open("../data/processed/data_card.json", "r") as f:
    card = json.load(f)

# Display dataset shapes
print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

# Display metadata keys
print("\nData card keys:", card.keys())

Train shape: (77455, 25)
Validation shape: (16597, 25)
Test shape: (16598, 25)

Data card keys: dict_keys(['dataset_name', 'doi', 'n_total', 'n_train', 'n_val', 'n_test', 'n_features', 'features', 'targets', 'split_strategy', 'sample_weight_col', 'random_state', 'cv_strategy', 'preprocessing'])


## 2. Inspect Dataset Columns

In [28]:
# Display all columns in the training dataset
print("Columns in dataset:\n")
print(train.columns.tolist())

Columns in dataset:

['PLT_CN', 'EXPN.ha_scaled', 'LAT_scaled', 'LON_scaled', 'eco.EXPN.ha_scaled', 'feat_carbon_sink', 'feat_eco3_target_mean', 'feat_eco_state_ratio', 'feat_enc_NA_L1CODE', 'feat_enc_NA_L3CODE', 'feat_enc_US_L4CODE', 'feat_enc_state', 'feat_expanded_per_ha', 'feat_growth_share', 'feat_gs_ratio', 'feat_lat_bin', 'feat_lat_x_eco1', 'feat_log_magnitude', 'feat_lon_bin', 'state.EXPN.ha_scaled', 'TPH.gs.dC.dN0.01', 'target_log', 'target_binary', 'target_class3', 'sample_weight']


## 3. Build a Leakage-Free Feature Set

In [29]:
# Select safer input features only
# These features do not directly include the target or obvious leakage columns
safe_features = [
    "EXPN.ha_scaled",
    "LAT_scaled",
    "LON_scaled",
    "eco.EXPN.ha_scaled",
    "feat_enc_NA_L1CODE",
    "feat_enc_NA_L3CODE",
    "feat_enc_US_L4CODE",
    "feat_enc_state",
    "state.EXPN.ha_scaled"
]

# Define the target column for 3-class classification
target_col = "target_class3"

# Create clean feature matrices and target vectors
X_train_clean = train[safe_features]
y_train_clean = train[target_col]

X_val_clean = val[safe_features]
y_val_clean = val[target_col]

X_test_clean = test[safe_features]
y_test_clean = test[target_col]

# Display shapes
print("X_train_clean shape:", X_train_clean.shape)
print("X_val_clean shape:", X_val_clean.shape)
print("X_test_clean shape:", X_test_clean.shape)

# Show class distribution
print("\nClass distribution in training set:")
print(y_train_clean.value_counts().sort_index())

X_train_clean shape: (77455, 9)
X_val_clean shape: (16597, 9)
X_test_clean shape: (16598, 9)

Class distribution in training set:
target_class3
0    25585
1    25507
2    26363
Name: count, dtype: int64


## 4. Train the Decision Tree Classifier

In [30]:
# Import the Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

# Create the model
# Hyperparameters are slightly restricted to reduce overfitting
dt_model_clean = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

# Train the model on the cleaned training data
dt_model_clean.fit(X_train_clean, y_train_clean)

print("Decision Tree model trained successfully.")

Decision Tree model trained successfully.


## 5. Make Predictions on Validation Data

In [31]:
# Predict class labels for the validation set
y_val_pred_clean = dt_model_clean.predict(X_val_clean)

# Show the first 10 predicted classes
print("First 10 predicted classes:", y_val_pred_clean[:10])

First 10 predicted classes: [0 2 1 0 2 2 2 0 2 0]


## 6. Evaluate the Model Performance

In [33]:
# Import evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Calculate validation accuracy
val_accuracy = accuracy_score(y_val_clean, y_val_pred_clean)

# Display model evaluation results
print("Validation Accuracy:", val_accuracy)

print("\nClassification Report:\n")
print(classification_report(y_val_clean, y_val_pred_clean))

print("Confusion Matrix:\n")
print(confusion_matrix(y_val_clean, y_val_pred_clean))

Validation Accuracy: 0.4435138880520576

Classification Report:

              precision    recall  f1-score   support

           0       0.44      0.49      0.46      5468
           1       0.42      0.29      0.34      5545
           2       0.46      0.55      0.50      5584

    accuracy                           0.44     16597
   macro avg       0.44      0.44      0.44     16597
weighted avg       0.44      0.44      0.44     16597

Confusion Matrix:

[[2693 1176 1599]
 [2016 1624 1905]
 [1451 1089 3044]]


## 7. Display Predictions as Low, Medium, and High

In [34]:
# Map numeric class labels to readable air quality categories
label_map = {
    0: "Low",
    1: "Medium",
    2: "High"
}

# Convert the first 10 actual and predicted labels into readable text
actual_labels = [label_map[x] for x in y_val_clean[:10]]
predicted_labels = [label_map[x] for x in y_val_pred_clean[:10]]

print("First 10 Actual Labels:   ", actual_labels)
print("First 10 Predicted Labels:", predicted_labels)

First 10 Actual Labels:    ['High', 'Low', 'Medium', 'Medium', 'High', 'High', 'Medium', 'High', 'Low', 'High']
First 10 Predicted Labels: ['Low', 'High', 'Medium', 'Low', 'High', 'High', 'High', 'Low', 'High', 'Low']


## 8. Final Observation

The initial model achieved 100% accuracy, which indicated possible data leakage. This happened because some engineered features were too closely related to the target variable. To fix this issue, a leakage-free feature set was selected by removing suspicious and target-related columns.

After rebuilding the model using safer input variables, the Decision Tree Classifier achieved a more realistic validation accuracy. This final result is considered more trustworthy for multiclass air quality classification into **Low**, **Medium**, and **High** categories.

## 9. Evaluate on the Test Set

In [35]:
# Predict class labels for the test set
y_test_pred_clean = dt_model_clean.predict(X_test_clean)

# Evaluate test performance
test_accuracy = accuracy_score(y_test_clean, y_test_pred_clean)

print("Test Accuracy:", test_accuracy)

print("\nTest Classification Report:\n")
print(classification_report(y_test_clean, y_test_pred_clean))

print("Test Confusion Matrix:\n")
print(confusion_matrix(y_test_clean, y_test_pred_clean))

Test Accuracy: 0.44800578382937706

Test Classification Report:

              precision    recall  f1-score   support

           0       0.44      0.50      0.47      5462
           1       0.41      0.29      0.34      5462
           2       0.48      0.55      0.51      5674

    accuracy                           0.45     16598
   macro avg       0.44      0.45      0.44     16598
weighted avg       0.44      0.45      0.44     16598

Test Confusion Matrix:

[[2737 1156 1569]
 [2006 1587 1869]
 [1464 1098 3112]]


### Model Limitations

The Decision Tree model shows moderate accuracy and struggles to clearly separate all three classes. This may be due to overlapping feature distributions and limited feature representation. Future improvements could include trying ensemble models, feature engineering, or hyperparameter tuning.